# Phase 2c: Merge weights (WiSE-FT) + final honest test

No training. Averages the base and fine-tuned weights into **one model** (same speed, same size).

1. Try mixes of 30% / 50% / 70% fine-tuned, **chosen on validation data only** (rule fixed before looking at test).
2. Refit the confidence temperature on validation.
3. Final score on the **full Jigsaw test set** (~64k comments), comparable to Detoxify's 0.9863.
4. Upload the merged model to your HF repo (replaces the main version; the old one stays in history).

**Settings:** GPU T4 ×2 · Internet On · Input: Toxic Comment Classification Challenge · Secret HF_TOKEN attached.
**Run with Save Version → Save & Run All** (~1.5 h, runs in background).

In [1]:
# ==== CONFIG ====
ALPHAS = [0.0, 0.3, 0.5, 0.7, 1.0]   # share of fine-tuned weights; 0 = base, 1 = fine-tuned (for reference)
CANDIDATES = [0.3, 0.5, 0.7]         # the merged models we choose between (rule: best mean validation AUC)
N_VAL = 10000
FT_REPO = "Vinay57/laya-jigsaw-moderation"
BASE_REPO = "convaiinnovations/laya"
PUSH = True
BATCH = 64

In [2]:
!pip install -q -U laya
import os, glob, json, time, shutil, numpy as np, pandas as pd, torch, laya
print("laya", laya.__version__, "| CUDA", torch.cuda.is_available())

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.6/58.6 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.5/118.5 kB 3.8 MB/s eta 0:00:00
laya 0.3.20 | CUDA True


## 1. Data: validation (for choosing) and full test (for the final score)

In [3]:
from sklearn.model_selection import train_test_split
LABELS = ['toxic','severe_toxic','obscene','threat','insult','identity_hate']
QUESTIONS = {
  'toxic':         {'type':'noul','instructions':'Is this comment rude, disrespectful or unreasonable?'},
  'severe_toxic':  {'type':'noul','instructions':'Is this comment extremely hateful, aggressive or disrespectful?'},
  'obscene':       {'type':'noul','instructions':'Does this comment contain obscene or vulgar language?'},
  'threat':        {'type':'noul','instructions':'Does this comment threaten to harm someone?'},
  'insult':        {'type':'noul','instructions':'Does this comment insult a person?'},
  'identity_hate': {'type':'noul','instructions':'Does this comment attack someone for their identity (race, religion, gender, sexuality, etc.)?'},
}
MAX_CHARS = 1200

BASE = os.path.dirname(glob.glob('/kaggle/input/**/train.csv*', recursive=True)[0])
def load(name):
    for ext in ['.csv', '.csv.zip']:
        p = f'{BASE}/{name}{ext}'
        if os.path.exists(p): return pd.read_csv(p)
    raise FileNotFoundError(name)
train = load('train')
test = load('test').merge(load('test_labels'), on='id')
test = test[test['toxic'] != -1].reset_index(drop=True)
train = train.drop_duplicates('comment_text').reset_index(drop=True)
tr, va = train_test_split(train, test_size=0.1, random_state=42, stratify=train['toxic'])   # same split as Phase 1/2
val_df = va.sample(min(N_VAL, len(va)), random_state=11)
seen_idx = test.sample(10000, random_state=42).index        # the 10k test sample we already analysed
print("val", len(val_df), val_df[LABELS].sum().to_dict())
print("test", len(test), test[LABELS].sum().to_dict())

val 10000 {'toxic': 971, 'severe_toxic': 98, 'obscene': 537, 'threat': 26, 'insult': 505, 'identity_hate': 99}
test 63978 {'toxic': 6090, 'severe_toxic': 367, 'obscene': 3691, 'threat': 211, 'insult': 3427, 'identity_hate': 712}


## 2. Load both models and set up direct full-precision scoring

In [4]:
from huggingface_hub import snapshot_download
from kaggle_secrets import UserSecretsClient
from safetensors.torch import load_file, save_file
from transformers import AutoTokenizer
from laya.agent import _fix_tokenizer_config
from laya.common import build_model, build_sequence, QTYPES, ece_score

HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
NOUL = QTYPES["noul"]
IGN = ["multilingual/*", "typed-decisions/*", "eval/*", "assets/*"]

ft_dir = snapshot_download(FT_REPO, token=HF_TOKEN, ignore_patterns=IGN)
base_dir = snapshot_download(BASE_REPO, token=HF_TOKEN, ignore_patterns=IGN)
for d in (ft_dir, base_dir): _fix_tokenizer_config(d)
tok = AutoTokenizer.from_pretrained(os.path.join(ft_dir, "tokenizer"))
cfg = json.load(open(os.path.join(ft_dir, "rl_agent_config.json")))

ft_sd = load_file(os.path.join(ft_dir, "model.safetensors"))
base_sd = load_file(os.path.join(base_dir, "model.safetensors"))
assert set(ft_sd) == set(base_sd), "weight names differ"

model = build_model(cfg, encoder_dir=os.path.join(ft_dir, "encoder")).to("cuda").eval()

def merged_weights(a):
    return {k: ((1 - a) * base_sd[k].float() + a * ft_sd[k].float()) if ft_sd[k].is_floating_point() else ft_sd[k]
            for k in ft_sd}

def collate(items, pad_id):
    n, L = len(items), max(len(it[0]) for it in items)
    kmax = max(len(it[1]) for it in items)
    ids = torch.full((n, L), pad_id, dtype=torch.long); att = torch.zeros((n, L), dtype=torch.long)
    mpos = torch.zeros((n, kmax), dtype=torch.long); mmask = torch.zeros((n, kmax), dtype=torch.bool)
    for i, (seq, mk) in enumerate(items):
        ids[i, :len(seq)] = torch.tensor(seq); att[i, :len(seq)] = 1
        mpos[i, :len(mk)] = torch.tensor(mk); mmask[i, :len(mk)] = True
    return ids, att, mpos, mmask, torch.full((n,), NOUL, dtype=torch.long)

def build_all(df):
    seqs = []
    for i, text in enumerate(df.comment_text.tolist()):
        state = {'comment': text[:MAX_CHARS]}
        for j, l in enumerate(LABELS):
            ids, mk = build_sequence(tok, state, {"t": "noul", "ins": QUESTIONS[l]["instructions"], "crit": {}},
                                     cfg["max_len"], cfg["head_max_len"])
            seqs.append((i, j, ids, mk))
    return seqs

@torch.no_grad()
def score_all(seqs, n_rows):
    order = sorted(range(len(seqs)), key=lambda k: len(seqs[k][2]))
    margin = np.zeros((n_rows, len(LABELS)))
    torch.cuda.synchronize(); t0 = time.time()
    for b in range(0, len(order), BATCH):
        idx = order[b:b + BATCH]
        ids, att, mpos, mmask, qt = collate([(seqs[k][2], seqs[k][3]) for k in idx], tok.pad_token_id)
        with torch.autocast("cuda", dtype=torch.float16):
            logits, _ = model(ids.cuda(), att.cuda(), mpos.cuda(), mmask.cuda(), qt.cuda())
        lg = logits.float()[:, :2].cpu().numpy()          # [false, true]
        for r, k in enumerate(idx):
            margin[seqs[k][0], seqs[k][1]] = lg[r, 1] - lg[r, 0]
    torch.cuda.synchronize()
    return margin, time.time() - t0

from sklearn.metrics import roc_auc_score
def aucs(y, m):
    a = {l: roc_auc_score(y[:, j], m[:, j]) for j, l in enumerate(LABELS)}
    a["mean"] = float(np.mean(list(a.values())))
    return a

Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

## 3. Choose the mix on validation only

In [5]:
t0 = time.time(); val_seqs = build_all(val_df); print(f"val sequences built in {time.time()-t0:.0f}s")
yv = val_df[LABELS].values
val_margins, val_rows = {}, {}
for a in ALPHAS:
    model.load_state_dict(merged_weights(a), strict=True)
    m, secs = score_all(val_seqs, len(val_df))
    val_margins[a] = m
    val_rows[f"merge a={a}"] = aucs(yv, m)
    print(f"a={a}: mean val AUC {val_rows[f'merge a={a}']['mean']:.4f} ({len(val_df)/secs:.1f} comments/sec)")

zn = lambda m: (m - m.mean(0)) / m.std(0)
val_rows["output blend (2 models)"] = aucs(yv, zn(val_margins[0.0]) + zn(val_margins[1.0]))
val_tab = pd.DataFrame(val_rows).T
print("\nVALIDATION (used for choosing):\n", val_tab.round(4).to_string())

best_a = max(CANDIDATES, key=lambda a: val_rows[f"merge a={a}"]["mean"])
print(f"\nChosen by rule (best mean validation AUC among {CANDIDATES}): a = {best_a}")

val sequences built in 36s


W0925 21:06:16.688000 23 torch/_inductor/utils.py:1679] [1/0_1] Not enough SMs to use max_autotune_gemm mode


a=0.0: mean val AUC 0.9817 (18.4 comments/sec)
a=0.3: mean val AUC 0.9891 (18.6 comments/sec)
a=0.5: mean val AUC 0.9901 (18.7 comments/sec)
a=0.7: mean val AUC 0.9899 (18.5 comments/sec)
a=1.0: mean val AUC 0.9839 (18.6 comments/sec)

VALIDATION (used for choosing):
                           toxic  severe_toxic  obscene  threat  insult  identity_hate    mean
merge a=0.0              0.9787        0.9830   0.9799  0.9864  0.9837         0.9786  0.9817
merge a=0.3              0.9865        0.9877   0.9927  0.9912  0.9881         0.9882  0.9891
merge a=0.5              0.9880        0.9894   0.9937  0.9928  0.9890         0.9877  0.9901
merge a=0.7              0.9884        0.9901   0.9937  0.9881  0.9894         0.9895  0.9899
merge a=1.0              0.9877        0.9881   0.9927  0.9552  0.9897         0.9899  0.9839
output blend (2 models)  0.9877        0.9879   0.9935  0.9877  0.9883         0.9916  0.9894

Chosen by rule (best mean validation AUC among [0.3, 0.5, 0.7]): a = 0.5

## 4. Refit confidence temperature on validation

In [6]:
from scipy.optimize import minimize_scalar
def fit_temp(margin, y):
    m, t = margin.ravel(), y.ravel().astype(float)
    def nll(T):
        p = np.clip(1 / (1 + np.exp(-m / T)), 1e-7, 1 - 1e-7)
        return -(t * np.log(p) + (1 - t) * np.log(1 - p)).mean()
    return float(minimize_scalar(nll, bounds=(0.2, 10), method="bounded").x)

T = fit_temp(val_margins[best_a], yv)
print(f"Fitted temperature: {T:.3f}")

Fitted temperature: 1.103


## 5. Final score on the full test set (never used for choosing)

In [7]:
model.load_state_dict(merged_weights(best_a), strict=True)
t0 = time.time(); test_seqs = build_all(test); print(f"test sequences built in {time.time()-t0:.0f}s")
tm, secs = score_all(test_seqs, len(test))
yt = test[LABELS].values
prob = 1 / (1 + np.exp(-tm / T))
p, yy = prob.ravel(), yt.ravel()
ece = ece_score(np.maximum(p, 1 - p), ((p >= 0.5) == (yy == 1)).astype(float))

full = aucs(yt, tm)
unseen = ~test.index.isin(seen_idx)
held = aucs(yt[unseen], tm[unseen])

ref = pd.DataFrame({
  "TF-IDF":   {"toxic": .9643, "severe_toxic": .9831, "obscene": .9770, "threat": .9920, "insult": .9703, "identity_hate": .9850, "mean": .9786},
  "Detoxify": {"toxic": .9739, "severe_toxic": .9915, "obscene": .9824, "threat": .9965, "insult": .9805, "identity_hate": .9928, "mean": .9863},
})
ref[f"Laya merged a={best_a}"] = pd.Series(full)
ref["(same, 54k never-seen part only)"] = pd.Series(held)
print("FULL TEST SET (%d comments):\n" % len(test), ref.round(4).to_string())
print(f"\nECE {ece:.4f} | speed {len(test)/secs:.1f} comments/sec (T4, batched, 6 questions/comment)")

pd.DataFrame(tm, columns=LABELS).to_csv("/kaggle/working/merged_test_margins.csv", index=False)
json.dump({"alpha": best_a, "temperature": T, "full_test": full, "unseen_54k": held, "ece": float(ece),
           "comments_per_sec": len(test) / secs, "validation": val_tab.round(5).to_dict()},
          open("/kaggle/working/final_results.json", "w"), indent=2)

test sequences built in 243s
FULL TEST SET (63978 comments):
                TF-IDF  Detoxify  Laya merged a=0.5  (same, 54k never-seen part only)
toxic          0.9643    0.9739             0.9726                            0.9730
severe_toxic   0.9831    0.9915             0.9887                            0.9890
obscene        0.9770    0.9824             0.9828                            0.9829
threat         0.9920    0.9965             0.9962                            0.9968
insult         0.9703    0.9805             0.9813                            0.9815
identity_hate  0.9850    0.9928             0.9902                            0.9902
mean           0.9786    0.9863             0.9853                            0.9856

ECE 0.0223 | speed 18.5 comments/sec (T4, batched, 6 questions/comment)


## 6. Save merged model, check it loads with official Laya, upload

In [8]:
OUT = "/kaggle/working/laya_jigsaw_merged"
os.makedirs(OUT, exist_ok=True)
save_file({k: v.half().contiguous() for k, v in merged_weights(best_a).items()}, os.path.join(OUT, "model.safetensors"))
for sub in ("encoder", "tokenizer"):
    shutil.copytree(os.path.join(ft_dir, sub), os.path.join(OUT, sub), dirs_exist_ok=True)
cfg_out = dict(cfg); cfg_out["temperature"] = [1.2, 1.2, T]
cfg_out["model_name"] = "laya-jigsaw-moderation-merged"; cfg_out["merge_alpha"] = best_a
cfg_out.pop("temperature_by_options", None)
json.dump(cfg_out, open(os.path.join(OUT, "rl_agent_config.json"), "w"), indent=2)

agent = laya.Agent(OUT, device="cuda")
chk = agent.predict_batch([{'comment': t[:MAX_CHARS]} for t in test.comment_text.iloc[:100]], QUESTIONS, batch_size=16)
ag = np.array([[float(r["answers"][l]["noul"]) for l in LABELS] for r in chk])
print(f"Official Agent vs our scoring, max diff: {np.abs(ag - prob[:100]).max():.4f} (small = fine)")

if PUSH:
    from huggingface_hub import HfApi
    api = HfApi(token=HF_TOKEN)
    card = f"""---
license: apache-2.0
base_model: convaiinnovations/laya
tags: [laya, toxicity, moderation, calibrated-decisions, wise-ft]
---
# Laya for toxic comment moderation (Jigsaw)

Answers 6 yes/no questions per comment (toxic, severe_toxic, obscene, threat, insult, identity_hate) with calibrated probabilities.
Fine-tuned on 12k balanced Jigsaw comments, then weight-merged with the base model (WiSE-FT, {int(best_a*100)}% fine-tuned; mix chosen on validation only).

| Model | Mean ROC-AUC, full Jigsaw test set ({len(test):,} comments) |
|---|---|
| **This model** | **{full['mean']:.4f}** |
| Detoxify original | 0.9863 |
| TF-IDF + logistic regression | 0.9786 |

ECE {ece:.4f} · ~{len(test)/secs:.0f} comments/sec on one T4 · $0/request self-hosted.
"""
    open(os.path.join(OUT, "README.md"), "w").write(card)
    api.upload_folder(folder_path=OUT, repo_id=FT_REPO,
                      commit_message=f"WiSE-FT merge a={best_a}: full-test mean AUC {full['mean']:.4f}")
    print(f"Uploaded to https://huggingface.co/{FT_REPO}")

Official Agent vs our scoring, max diff: 0.0028 (small = fine)


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Uploaded to https://huggingface.co/Vinay57/laya-jigsaw-moderation
